# Week 7 · Saving and history

[![Open in Colab](../assets/badges/colab-badge.svg)](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_07.ipynb)

A system that forgets cannot be trusted. This week your system learns to save its records, load them back, and keep a history that explains what happened.

| | |
|---|---|
| **This week's question** | How does a system remember, and how can it explain what happened? |
| **Python you will learn** | `open` and `with`, `"w"` and `"a"`, `os.path.exists`, `json.dumps` / `json.loads`, `json.dump` / `json.load`, JSON Lines, slices such as `events[-5:]` |
| **Principle 7** | **Never lose data.** Save safely, keep a history, and be able to explain what happened. |
| **File you add** | `my_foodbridge/storage.py` → `save_records`, `load_records`, `append_event`, `read_events` |
| **What you will see** | `bridge.show("history")`: the sample day saved with your code, the two files, a timeline of events, and a restart that loads everything back |

*Campus FoodBridge is fictional. All organisations, people and numbers are invented.*

![System map, week 7: storage.py saves safely and keeps a history](../assets/diagrams/system_map_w07.svg)

### The story so far

Your folder holds four files: `rules.py` and `records.py` (Week 4), `data_model.py`, which checks every row at the door (Week 5), and `lifecycle.py`, which allows only sensible moves (Week 6). Your **code** is safe: the folder lives in your Google Drive.

Your **data** is not. Yesterday at 4 pm Colab restarted. Every offer, every approval and every delivery of the day lived only in the program's memory, and now it is gone. Sunrise Bakery phones: *"What happened to our 30 portions of bread rolls?"* The coordinator cannot say. There is no file to open and no history to read.

A system also has to **remember**: save its records safely, and keep a history it can explain. That is this week's work.

## Before you start

1. **File → Save a copy in Drive.** Work in your copy.
2. Run the setup cell. In Colab it asks to connect Google Drive: say yes, so your `my_foodbridge/` folder is kept between weeks.
3. ⭐ = everyone · ⭐⭐ = stretch.

In [ ]:
# ▶ Run this cell first (setup)
# In Colab or Jupyter, this cell unpacks the FoodBridge kit next to the notebook.
# This web page is for reading. To run the notebook, use the 'Open in Colab' button at the top of this page.


## Class warm-up · Research 5 exchange (10 min)

In groups of three, share what you found about keeping data safe.

1. Give one real example of a **backup**, an **audit log** or a **transaction** from the system you studied.
2. What happened, or could happen, when data was lost? Who was hurt?
3. Who should be allowed to read the history, and who should not?

📝 **My notes:** one idea from a classmate that I want to remember …

## Session 1 · Learn and try (90 min)

### The idea: memory forgets, records must not

Think of a small shop. During the day the owner writes sales on a whiteboard. At night the board is wiped clean. A careful owner also keeps a **sales book**: every sale is a new line in ink, with the time. A mistake is fixed with a new line ("refund, 2 pm"), never by tearing out a page.

A program's memory is the whiteboard: your lists and dictionaries disappear when the program stops or Colab restarts. An information system needs two more things:

- **Persistence:** the records are saved in a file, so they are still there tomorrow.
- **An audit history:** a list of events that says **who did what, and when**. We only ever add to it (it is **append-only**). Then, when a donor asks *"what happened to my bread rolls?"*, someone can answer.

**Where do files live?** The practice files in Session 1 go in the notebook's own folder. They disappear when the runtime stops, and that is fine for practice. Your system's files go in `my_foodbridge/`, which the setup cell keeps in your Google Drive, so they survive a restart. Click the folder icon on the left of Colab (the **Files** panel) to see both.

### 1 · Write a file, then read it line by line

`open` connects your program to a file. `with` closes the file for you at the end of the block, even when something goes wrong. `"w"` means *write*. Without a mode, `open` only reads. A `for` loop over a file gives you one line at a time.

🤔 **Predict** how many lines the loop prints, and what `line.strip()` removes.

In [ ]:
with open("shopping.txt", "w", encoding="utf-8") as file:
    file.write("rice\n")
    file.write("eggs\n")

with open("shopping.txt", encoding="utf-8") as file:
    for line in file:
        print("-", line.strip())

**Prediction:** …  **Result:** …

`"\n"` ends a line. `write` does not add it for you, and `strip()` removes it again when you read. `encoding="utf-8"` lets the file hold any language, including Khmer.

### ✍️ Try it · `read_lines`

Write `read_lines(path)`: return a list with one item per line of the file, without the `"\n"` at the end.

In [ ]:
with open("class_list.txt", "w", encoding="utf-8") as file:   # a second file to test with
    file.write("Dara\nSokha\nVanna\n")


def read_lines(path):
    lines = []
    # TODO: open the file at path; for each line, append line.strip() to lines
    return lines


bridge.check("read_lines", read_lines, examples=[
    ("shopping.txt", ["rice", "eggs"]),
    ("class_list.txt", ["Dara", "Sokha", "Vanna"]),
])

### 2 · `"w"` empties the file, `"a"` adds to the end

Opening a file in the wrong mode is one of the most common ways programs lose data.

🤔 **Predict** what the last part prints. The file starts with two lines, *maths* and *English*.

In [ ]:
with open("homework.txt", "w", encoding="utf-8") as file:
    file.write("maths\n")
    file.write("English\n")

with open("homework.txt", "w", encoding="utf-8") as file:     # "w" again
    file.write("history\n")

with open("homework.txt", "a", encoding="utf-8") as file:     # "a"
    file.write("Khmer\n")

with open("homework.txt", encoding="utf-8") as file:
    print(file.read())

**Prediction:** …  **Result:** …  **Why:** …

`"w"` empties the file **the moment it opens**, before you write anything. *maths* and *English* are gone, and Python gives no warning. `"a"` (*append*) keeps everything and adds to the end.

### ✍️ Try it · `add_sale`

The shop's sales book starts the day with one sale. Write `add_sale(path, sale)`: add `sale` as a new line at the **end** of the file. The last two lines are given: they count the lines now in the file, so the check can see whether the old sales were kept.

In [ ]:
with open("sales_book.txt", "w", encoding="utf-8") as file:   # the day starts with one sale
    file.write("08:00 two bread rolls\n")


def add_sale(path, sale):
    # TODO: open path with "a" (and encoding="utf-8") and write sale + "\n"

    with open(path, encoding="utf-8") as file:    # given: count the lines now in the file
        return len(file.readlines())              # readlines() gives a list, one item per line


bridge.check("add_sale", add_sale, examples=[
    (("sales_book.txt", "09:15 one rice box"), 2),
    (("sales_book.txt", "10:30 three bananas"), 3),
])

### 3 · Is the file there? `os.path.exists`

Reading a file that does not exist stops the program with `FileNotFoundError`. On the first day of a new system there is no file yet. That is normal, not an error, so check first.

In [ ]:
import os

print(os.path.exists("shopping.txt"))
print(os.path.exists("diary.txt"))

The pattern you will use all week: **if the file exists, read it; if not, start with an empty list.**

### 4 · JSON: lists and dictionaries as text

A file holds only text, but your records are lists and dictionaries. **JSON** is a standard way to write them as text. People can read it, and almost every programming language can load it. `json.dumps` turns data into JSON text; `json.loads` turns JSON text back into data. (The `s` means *string*.)

🤔 **Predict** the last line: is `back["credit_usd"]` text or a number?

In [ ]:
import json

phone = {"owner": "Dara", "credit_usd": 5, "plans": ["data", "calls"], "active": True}
text = json.dumps(phone)
print(text)
print(type(text))

back = json.loads(text)
print(back["credit_usd"] + 1)

**Prediction:** …  **Result:** …

In Week 5 every value from the CSV file arrived as text, so you needed `int()`. JSON keeps numbers as numbers (and writes `True` as `true`). That is one reason it is a good format for records.

### ✍️ Try it · `top_up`

A phone shop keeps each account as JSON text. Write `top_up(text, amount)`: load the account, add `amount` to its `"credit_usd"`, and return the new JSON text.

In [ ]:
def top_up(text, amount):
    # TODO 1: account = json.loads(text)
    # TODO 2: add amount to account["credit_usd"]
    # TODO 3: return json.dumps(account)
    return text


bridge.check("top_up", top_up, examples=[
    (('{"owner": "Dara", "credit_usd": 5}', 2), '{"owner": "Dara", "credit_usd": 7}'),
    (('{"owner": "Sokha", "credit_usd": 0}', 10), '{"owner": "Sokha", "credit_usd": 10}'),
])

### 5 · `json.dump` and `json.load` work with a file

Without the `s`, the same two functions write to a file and read from a file. `indent=2` puts each item on its own line, so a person can read the file too.

In [ ]:
classes = [{"code": "MIS101", "students": 42}, {"code": "ACC102", "students": 35}]

with open("classes.json", "w", encoding="utf-8") as file:
    json.dump(classes, file, indent=2)

with open("classes.json", encoding="utf-8") as file:
    loaded = json.load(file)

print(loaded[1]["students"])
print(loaded == classes)

Open `classes.json` in the Files panel (double-click it) and look: it is plain text.

### ✍️ Try it · `load_list`

Write `load_list(path)`: when there is no file at `path`, return `[]`; otherwise return the list saved in the file. This is how a new system starts on its first day, and it is almost your `load_records`.

In [ ]:
def load_list(path):
    # TODO 1: if there is no file at path, return []   (use os.path.exists)
    # TODO 2: otherwise open the file and return json.load(file)
    return None


bridge.check("load_list", load_list, examples=[
    ("classes.json", [{"code": "MIS101", "students": 42}, {"code": "ACC102", "students": 35}]),
    ("no_such_file.json", []),
])

### 6 · JSON Lines: a history you only add to

A history has a different shape from a list of records. We never rewrite it; we **add one line per event**. Each line is one small JSON object. This is called **JSON Lines**, and the files end in `.jsonl`.

In [ ]:
def log_event(path, event):
    with open(path, "a", encoding="utf-8") as file:     # "a": add to the end, never empty
        file.write(json.dumps(event) + "\n")             # one event = one line


log_event("phone_history.jsonl", {"time": "09:00", "event": "top up", "usd": 5})
log_event("phone_history.jsonl", {"time": "09:30", "event": "bought data", "usd": 3})

with open("phone_history.jsonl", encoding="utf-8") as file:
    for line in file:
        event = json.loads(line)
        print(event["time"], event["event"], event["usd"])

🤔 **Predict, then try:** run the cell above a second time. What happens to the history? Is that right for a history? Would it be right for a list of records?

**Prediction:** …  **Result:** …  **Why:** …

### 7 · Slices: the last few items

A history grows all day, but a screen has room for only a few lines. Square brackets with a colon take a **slice**, a part of a list (or a text). A minus sign counts from the end.

🤔 **Predict** the last line: what are "the last 10" of only 7 items?

In [ ]:
calls = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print(calls[:2])      # the first 2
print(calls[-3:])     # the last 3
print(calls[-10:])

**Prediction:** …  **Result:** …

A slice never stops your program, even when the list is shorter than you asked for. It also builds a **new** list: `calls` does not change.

### ✍️ Try it · `last_five`

The coordinator's screen shows the newest five events of the history. Write `last_five(events)`: return the last 5 items of the list (all of them when there are fewer).

In [ ]:
def last_five(events):
    # TODO: return a slice with the last 5 items of events
    return []


bridge.check("last_five", last_five, examples=[
    (["#1", "#2", "#3", "#4", "#5", "#6", "#7"], ["#3", "#4", "#5", "#6", "#7"]),
    (["#1", "#2"], ["#1", "#2"]),
    ([], []),
])

### ✍️ Try it · `count_events` (FoodBridge)

In FoodBridge, each event in the history is a dictionary. Its `"event"` key says what kind of event it was: `"added"`, `"input_rejected"` or `"moved"`. Write `count_events(events)`: return a dictionary `{kind of event: how many}`. It is the Week 3 dictionary accumulator.

In [ ]:
def count_events(events):
    counts = {}
    # TODO: for each event, add 1 to counts[event["event"]]   (counts.get(kind, 0) + 1)
    return counts


short_day = [
    {"step": 1, "event": "added", "offer": "F001"},
    {"step": 2, "event": "added", "offer": "F002"},
    {"step": 3, "event": "input_rejected", "offer": ""},
    {"step": 4, "event": "moved", "offer": "F002"},
]
bridge.check("count_events", count_events, examples=[
    (short_day, {"added": 2, "input_rejected": 1, "moved": 1}),
    ([], {}),
])

### ✍️ Try it · `story_of` (FoodBridge)

Sunrise Bakery phones: *"What happened to our bread rolls?"* Write `story_of(events, offer_id)`: return the `"detail"` of every event about that offer, oldest first. This is what an audit history is for.

In [ ]:
def story_of(events, offer_id):
    details = []
    # TODO: for each event whose "offer" is offer_id, append event["detail"] to details
    return details


bakery_day = [
    {"step": 1, "event": "added", "offer": "F001", "detail": "30 portions of Bread rolls"},
    {"step": 2, "event": "added", "offer": "F002", "detail": "18 portions of Fried rice boxes"},
    {"step": 3, "event": "moved", "offer": "F001", "detail": "new -> approved for Hope Children's Centre"},
    {"step": 4, "event": "moved", "offer": "F001", "detail": "approved -> picked_up"},
]
bridge.check("story_of", story_of, examples=[
    ((bakery_day, "F001"), ["30 portions of Bread rolls", "new -> approved for Hope Children's Centre",
                            "approved -> picked_up"]),
    ((bakery_day, "F002"), ["18 portions of Fried rice boxes"]),
    ((bakery_day, "F999"), []),
])

## Session 2 · Practise and judge (90 min)

*If you opened this notebook in a new session, run the setup cell at the top first so `bridge` is ready.*

### 🎫 This week's ticket

> **From:** the coordinator · **To:** you
> "Yesterday Colab restarted at 4 pm and the whole day was gone. Sunrise Bakery phoned about its 30 bread rolls, and I could not say what had happened. I need the system to save every change, load it all back in the morning, and keep a history I can read."

| | |
|---|---|
| **File** | `my_foodbridge/storage.py` |
| **Setting** | `LOG_EVENTS`: the kinds of event the history keeps (default `["added", "input_rejected", "moved"]`) |
| **Functions** | `save_records(records, path)` · `load_records(path)` · `append_event(path, event)` · `read_events(path)` |
| **How the system uses them** | after every change it calls your `save_records`; for every event in `LOG_EVENTS` it calls your `append_event`; when it starts, it calls your `load_records` and `read_events` |
| **Done when** | the four checks pass, and `bridge.show("history")` shows your four parts ✅ and a restart test that gets the same 4 records back |

*Watch the parameter order: `save_records(records, path)` takes the data first, while `append_event(path, event)` takes `path` first.*

| Call | What must happen |
|---|---|
| `save_records(records, "offers.json")` | the file holds exactly that list; saving again **replaces** it |
| `load_records("offers.json")` | the same list back, numbers still numbers |
| `load_records("nothing_saved.json")` | `[]` (no file yet is normal) |
| `append_event("history.jsonl", event)`, twice | two lines in the file; the first line is kept |
| `read_events("history.jsonl")` | every event, oldest first |
| `read_events("nothing_logged.jsonl")` | `[]` |

### ✏️ Model it on paper first

Draw two boxes on paper: `offers.json` and `history.jsonl`. Then three things happen:

1. F001, 30 portions of bread rolls, is added.
2. A row is refused at the door: "portions must be a whole number".
3. F001 is approved for Hope Children's Centre.

After each step, draw what is inside each file. Then answer: Which file is **replaced** each time? Which one only **grows**?

In [ ]:
# 📝 Describe your drawing here (or paste a photo into a text cell).

### 🧩 Ladder 1 · Put the lines in order

The volunteer team keeps a sign-in history as JSON Lines. Click the lines into order. One line does not belong.

In [ ]:
bridge.parsons("sign_in_history")

<details><summary>Show the puzzle answer (if reading on the website)</summary>

```python
def read_sign_ins(path):
    sign_ins = []
    if not os.path.exists(path):
        return sign_ins
    with open(path, encoding="utf-8") as file:
        for line in file:
            sign_ins.append(json.loads(line))
    return sign_ins
```

The distractor line `with open(path, "w", encoding="utf-8") as file:` does not belong: `"w"` empties the file the moment it opens, which would erase every sign-in.

</details>

### 🧩 Ladder 2 · Fill the blanks

The phone shop keeps every top-up as one JSON line. `total_top_ups(path)` returns the total in USD, or `0` when there is no file yet. Replace each `____`.

In [ ]:
import json
import os

top_ups = [
    {"time": "08:10", "owner": "Dara", "usd": 5},
    {"time": "12:40", "owner": "Dara", "usd": 2},
    {"time": "19:05", "owner": "Dara", "usd": 3},
]
with open("top_ups.jsonl", "w", encoding="utf-8") as file:     # a small history to test with
    for entry in top_ups:
        file.write(json.dumps(entry) + "\n")


def total_top_ups(path):
    total = 0
    if not ____(path):                      # TODO: which os.path function says whether a file is there?
        return total
    with open(path, encoding="utf-8") as file:
        for line in file:
            entry = ____(line)               # TODO: which json function turns one line of text into a dictionary?
            total = total + entry[____]      # TODO: which key holds the amount?
    return total


bridge.check("total_top_ups", total_top_ups, examples=[
    ("top_ups.jsonl", 10),
    ("no_top_ups.jsonl", 0),
])

### 🧩 Ladder 3 · From a blank cell

The coordinator wants to see why rows were refused at the door today. Write `refused_rows(path)`: read the FoodBridge history file at `path` and return the `"detail"` of every `"input_rejected"` event, oldest first. Return `[]` when there is no file.

In [ ]:
import json
import os

practice_day = [
    {"step": 1, "event": "added", "offer": "F001", "detail": "30 portions of Bread rolls"},
    {"step": 2, "event": "input_rejected", "offer": "", "detail": "portions must be a whole number, for example 12."},
    {"step": 3, "event": "input_rejected", "offer": "", "detail": "donor is required."},
    {"step": 4, "event": "moved", "offer": "F001", "detail": "new -> approved for Hope Children's Centre"},
]
with open("practice_history.jsonl", "w", encoding="utf-8") as file:   # a short history to test with
    for event in practice_day:
        file.write(json.dumps(event) + "\n")


def refused_rows(path):
    # Your code here.
    return None


bridge.check("refused_rows", refused_rows, examples=[
    ("practice_history.jsonl", ["portions must be a whole number, for example 12.", "donor is required."]),
    ("no_history.jsonl", []),
])

### ⚖️ Good or risky?

You asked an AI assistant: *"Write a function that saves the day's offers and the history at the end of the day. Keep the files small."* It gave you **Version A**. **Version B** uses this week's pieces. On a good day, both run without errors.

**Version A** (from the AI assistant)

```python
def save_day(records, events):
    with open("offers.json", "w") as file:
        json.dump(records, file)
    # keep the history small: only the last 10 events
    with open("history.jsonl", "w") as file:
        for event in events[-10:]:
            file.write(json.dumps(event) + "\n")
```

**Version B**

```python
def save_day(records, new_events):
    temp_path = "offers.json.tmp"
    with open(temp_path, "w", encoding="utf-8") as file:   # 1. write the new data here
        json.dump(records, file, indent=2)
    os.replace(temp_path, "offers.json")                    # 2. swap it in, in one step
    with open("history.jsonl", "a", encoding="utf-8") as file:
        for event in new_events:
            file.write(json.dumps(event) + "\n")            # add to the end; never rewrite
```

![Safe save: write a temporary file, then replace the old file in one step](../assets/diagrams/safe_save.svg)

Use the sample day: F001 bread is picked up for Hope Children's Centre, F002 rice is delivered, F003 bananas were rejected ("no cold storage for fruit today"), and F004 curry is approved and waiting for the Old Market Elders' Group. The day has 16 events.

1. Which offer or person is harmed by Version A? Point at the line.
2. Look at Version B and the diagram: when does `offers.json.tmp` exist, and for how long? If the power cuts halfway through the save, what is left with each version? (Run the experiment below.)
3. How would a test catch each problem?

🔌 **Pull the plug.** Yesterday's two offers are saved. Today's save of three offers stops after 25 characters (`text[:25]` is a slice of the text). Run the cell and read the result.

In [ ]:
import json

yesterday = [{"id": "F001", "portions": 30}, {"id": "F002", "portions": 18}]
for path in ["risky.json", "safe.json"]:
    with open(path, "w", encoding="utf-8") as file:
        json.dump(yesterday, file)

today = yesterday + [{"id": "F003", "portions": 40}]
text = json.dumps(today)

# Version A writes straight into the file ... and the power goes off after 25 characters.
with open("risky.json", "w", encoding="utf-8") as file:
    file.write(text[:25])

# Version B writes the temporary file ... and the power goes off before os.replace.
with open("safe.json.tmp", "w", encoding="utf-8") as file:
    file.write(text[:25])

for path in ["risky.json", "safe.json"]:
    try:
        with open(path, encoding="utf-8") as file:
            print(path, "→", json.load(file))
    except ValueError:
        print(path, "→ broken: it cannot be loaded")

📝 **Problems I found in Version A** (for each: which line, and which offer or person is harmed):

1.
2.
3.

<details><summary>Instructor's view</summary>

1. **It writes straight over the only copy.** `open("offers.json", "w")` empties the file the moment it opens. If Colab disconnects or the power cuts halfway, the file is half-written and cannot be loaded at all (the experiment shows it). All four offers are lost, not only today's change: F004's 12 portions of curry, which the Old Market Elders' Group is still waiting for, and F001's bread on its way to Hope Children's Centre. Version B writes a complete temporary file first; if it stops, the old `offers.json` is still whole. At worst one change is lost, and the history shows what it was.
2. **It rewrites the history and throws old events away.** `"w"` plus `events[-10:]` keeps only the last 10 events. The sample day has 16, so the four "added" events and two refusals are gone today. After one more busy day, the reason F003 was rejected ("no cold storage for fruit today") is gone too. When Green Market Stall asks why its bananas were rejected, nobody can answer. A history must be append-only: Version B opens it with `"a"`.
3. **A rewritten history cannot be trusted.** Code that is allowed to rewrite the file can also change an old line, so nobody can show what really happened at 10:00.
4. Small point: no `encoding="utf-8"`. `json.dump` turns Khmer letters into codes such as `\u1780` by default, so this file survives, but the habit is risky: plain text in Khmer written without it may break on some computers.

**How a test catches it:** save twice and check that the first event is still in the history (the kit's `append_event` check does exactly this); stop a save halfway, as in the experiment, and check that the old file still loads.

**Is Version B perfect?** No. If someone calls it twice with the same `new_events`, the history gets the events twice. Good code still needs checks. AI code often "simplifies" by dropping exactly the lines that protect data on a bad day. Ask: *what happens on a bad day?*

</details>

## Session 3 · Build: `storage.py` (90 min)

*If you opened this notebook in a new session, run the setup cell at the top first so `bridge` and your `my_foodbridge/` folder are ready.*

### 📁 Your folder now

In [ ]:
bridge.show()

### Write your file

The cell below **is** your file. Its first line, `%%writefile my_foodbridge/storage.py`, saves the rest of the cell as that file. Run it again after every change.

You have practised every piece: `load_records` is your Try it `load_list`, `read_events` is the Ladder 1 puzzle plus one guard to skip blank lines (`if line.strip() != "":`), and `save_records` is the safe save from Version B.

Note: the **system** reads `LOG_EVENTS` before it calls `append_event` — your `append_event` should always write the event it is given.

In [ ]:
%%writefile my_foodbridge/storage.py
# storage.py · Week 7: never lose data, and keep a history
import json
import os

LOG_EVENTS = ["added", "input_rejected", "moved"]   # read by the system; append_event writes every event it is given


def save_records(records, path):
    """Save every record as JSON. Write a temporary file first, then swap it in."""
    temp_path = path + ".tmp"
    # TODO 1: open temp_path with "w" (encoding="utf-8") and json.dump(records, file, indent=2)
    # TODO 2: swap the new file in: os.replace(temp_path, path)
    return None


def load_records(path):
    """Return the saved records, or [] when nothing is saved yet."""
    # TODO 1: if there is no file at path, return []
    # TODO 2: open the file and return json.load(file)
    return []


def append_event(path, event):
    """Add one event to the end of the history file (one JSON line)."""
    # TODO: open path with "a" and write json.dumps(event) + "\n"
    return None


def read_events(path):
    """Return every event in the history file, oldest first."""
    # TODO 1: if there is no file at path, return []
    # TODO 2: read line by line; skip blank lines (if line.strip() != "":) and json.loads the rest
    return []

### Check each function

Each check works in a temporary folder of its own, so it never touches your real files.

In [ ]:
bridge.check("save_records", folder="my_foodbridge")
bridge.check("load_records", folder="my_foodbridge")

In [ ]:
bridge.check("append_event", folder="my_foodbridge")
bridge.check("read_events", folder="my_foodbridge")

### 👀 See it

`bridge.show("history")` runs the sample day with **your** `storage.py`. The system saves into `my_foodbridge/demo_day/`: `offers.json` (the records, replaced after every change) and `history.jsonl` (one line per event). Then a brand-new system starts, loads both files and compares: that is the **restart test**. A part that is not finished yet is replaced by the model version, and the picture says so.

In [ ]:
bridge.show("history")

Each line of `history.jsonl` records the step, the kind of event, the offer, the detail and the time. A real system would also record **who** made the change, such as the coordinator's name.

Your folder is in Google Drive, so these files stay safe even after Colab disconnects or deletes the runtime next week. Open the **Files** panel → `my_foodbridge` → `demo_day`, and double-click `history.jsonl` to read it.

**Try a real restart (optional):** choose **Runtime → Restart session** (this clears Python's memory), run the setup cell, then run the cell below (not the `show` cell: it makes the files again).

In [ ]:
import json
import os

path = "my_foodbridge/demo_day/offers.json"
history_path = "my_foodbridge/demo_day/history.jsonl"
if not os.path.exists(path):
    print("No saved file. Is your folder connected to Google Drive? Run the setup cell and bridge.show('history').")
else:
    with open(path, encoding="utf-8") as file:
        saved = json.load(file)
    print(len(saved), "records came back. The first:", saved[0]["id"], saved[0]["food"], saved[0]["status"])
    if not os.path.exists(history_path):
        print("No history file yet (check LOG_EVENTS).")
    else:
        with open(history_path, encoding="utf-8") as file:
            lines = [line.strip() for line in file if line.strip() != ""]
        print(len(lines), "events in the history.")
        if lines:
            print("The newest:", lines[-1])

### 🧭 Your decision · what does the history keep?

Every event in the history is a promise: *"we can explain this later."* That is **accountability**. But every event also has a cost: a longer history is harder to read, and the one event that matters is easier to miss (**noise**).

| Event | When it happens | What it records | If you drop it |
|---|---|---|---|
| `"added"` | an offer passes the door check | ID, portions, food | nobody can say when an offer arrived |
| `"input_rejected"` | a row is refused at the door | the reason | a donor whose row disappeared gets no answer; the same mistake can repeat unseen |
| `"moved"` | a status changes | old → new status, partner, note | nobody can say who got the food, or why F003 was rejected |

1. Change `LOG_EVENTS` at the top of your file. Keep at least one kind (or keep all three if you can explain why) so you still have a history to inspect. *(If you set `LOG_EVENTS = []`, no `history.jsonl` is written at all.)*
2. Run the file cell again, then the cell below. The check still passes, and the timeline changes.

In [ ]:
bridge.check("append_event", folder="my_foodbridge")
bridge.show("history")

🤔 How many events are in the timeline now? Which question can the coordinator no longer answer?

**My answer:** …

**Decision log** — *My choice:* `LOG_EVENTS = …` · *An alternative:* … · *Why I chose it:* … · *Who is affected / the tradeoff:* accountability ↔ noise: …

### ⭐⭐ Make one of your own

A sales book fixes a mistake with a **new line**, never an eraser. Ideas: (a) `add_correction(path, step, reason)` adds a `"corrected"` event instead of changing an old line; (b) `events_per_offer(events)` finds the offer with the longest story. The demo day's history is in `my_foodbridge/demo_day/history.jsonl`.

In [ ]:
# ⭐⭐ Your own function, and a line that tries it.

### 📁 Your folder now

In [ ]:
bridge.show()

## Finish

- [ ] I shared my Research 5 finding in the warm-up.
- [ ] I can explain why `open(path, "w")` is risky, and what the temporary file in `save_records` protects.
- [ ] My `storage.py` passes all four checks, and `bridge.show("history")` passes the restart test.
- [ ] My decision log explains `LOG_EVENTS`: what my history keeps, and what it loses.

### Research 6 · Checking AI output (5%)

AI assistants answer quickly and sound sure, even when they are wrong. In this task you practise checking.

1. Ask an AI assistant a **factual or technical question** you care about (Python, a rule at your university, a health fact, a price, a date). Or use the supplied example below.
2. Check each claim against an **original source**: official documentation, the rule itself, the organisation's own website, or a test you run yourself. Another AI answer does not count.
3. Make a short table: *claim · accept, correct or ask about · how I checked*.
4. In one or two sentences: what could have gone wrong if you had trusted the answer without checking?

**Supplied example.** Question: *"In Python, what happens when I save to a file that already exists?"* An AI assistant answered:

> 1. `open(path, "w")` keeps the old text and adds your new text at the end.
> 2. `open(path, "a")` adds new text at the end of the file.
> 3. `json.dump` can save any Python value, including dates.
> 4. `os.replace` swaps in the new file in one step, and it works the same way on every computer.

Hint: you can test claims 1–3 in a notebook cell in a few minutes. For claim 4, read the Python documentation for `os.replace`.

Use at least one source and name it. Bring your table to the **Week 8 class warm-up**. Submit by email to submit.progmis@gmail.com; the deadline and file name will be announced in class.

**Next week:** the same records, shown to different people: who may see which fields, and how do you keep the numbers honest? You add `my_foodbridge/report.py` and draw your first chart.

⬅ **Previous:** [Week 6 · Lifecycles](Week_06.ipynb) · **Next:** [Week 8 · Reports, screens and a first chart](Week_08.ipynb) ➡